# Day 16 — The p-value
> What it means, what it doesn't mean, and how to calculate it.

## Definition

> **p-value** = The probability of observing a test statistic at least as extreme as the one computed from the sample, **assuming H₀ is true**.

### Common Misconceptions
| ❌ Wrong | ✅ Correct |
|----------|----------|
| p-value is the probability H₀ is true | p-value assumes H₀ is true |
| p < 0.05 means important | Statistical ≠ Practical significance |
| Large p means H₀ is proven | Absence of evidence ≠ evidence of absence |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns
sns.set_theme(style='whitegrid')
np.random.seed(42)

# Simulate: does a new landing page improve time-on-page?
control   = np.random.normal(loc=4.0, scale=1.2, size=150)   # minutes
treatment = np.random.normal(loc=4.5, scale=1.3, size=150)

t_stat, p_value = stats.ttest_ind(control, treatment)
print(f"t-statistic : {t_stat:.4f}")
print(f"p-value     : {p_value:.6f}")
print(f"Significant : {p_value < 0.05}")


In [ ]:
# Visualize p-value on the t-distribution
df = len(control) + len(treatment) - 2
x = np.linspace(-5, 5, 500)
y = stats.t.pdf(x, df=df)
crit = stats.t.ppf(0.975, df=df)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x, y, 'k-', lw=2, label='t-distribution (H₀)')
ax.fill_between(x, y, where=(x >= crit),  alpha=0.3, color='red', label=f'α/2 = 0.025')
ax.fill_between(x, y, where=(x <= -crit), alpha=0.3, color='red')
ax.axvline( t_stat, color='navy', lw=2, linestyle='--', label=f't = {t_stat:.3f}')
ax.axvline(-t_stat, color='navy', lw=2, linestyle='--')
ax.set_title(f'p-value = {p_value:.4f}  →  {"Reject H₀" if p_value < 0.05 else "Fail to Reject H₀"}',
             fontweight='bold')
ax.legend()
ax.set_xlabel('t-statistic'); ax.set_ylabel('Density')
plt.tight_layout()
plt.savefig('../results/02_pvalue.png', dpi=150)
plt.show()


## p-value vs Effect Size

With large samples, even trivial differences become significant. Always check **practical significance**.

In [ ]:
# Demonstrate: same effect, different sample sizes → different p-values
sample_sizes = [30, 100, 300, 1000, 3000]
results = []
for n in sample_sizes:
    a = np.random.normal(4.0, 1.2, n)
    b = np.random.normal(4.1, 1.2, n)   # tiny real difference
    _, p = stats.ttest_ind(a, b)
    results.append((n, round(p, 5)))

print(f"{'Sample size':>12} | {'p-value':>10} | {'Significant':>12}")
print('-' * 40)
for n, p in results:
    print(f"{n:>12} | {p:>10} | {'YES' if p < 0.05 else 'no':>12}")
